# 08  Real-World Validation
## ShopEase Europe | Sentiment Analysis Project 
**Objective:** Validate our trained models against genuine customer 
reviews from a real-world multilingual dataset, addressing the 
template-based limitation identified in the ShopEase dataset. This 
notebook tests whether our models generalise beyond the synthetic 
training data.

In [4]:
import os
import pandas as pd
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

Libraries loaded successfully


## Download Real-World Sample
Pulling a small sample of genuine customer reviews across our four 
core languages from the Amazon Multilingual Reviews dataset on 
Hugging Face.

In [9]:
languages = ['en', 'fr', 'de', 'es']
samples = []

for lang in languages:
    dataset = load_dataset('mteb/amazon_reviews_multi', lang, split='validation')
    dataset = dataset.shuffle(seed=42)
    lang_samples = dataset.select(range(20))
    for item in lang_samples:
        samples.append({
            'language': lang,
            'review_text': item['text'],
            'star_rating': item['label'] + 1
        })

real_world_df = pd.DataFrame(samples)
print(f"Downloaded {len(real_world_df)} real reviews across {len(languages)} languages")
print(f"\nStar rating distribution:")
print(real_world_df['star_rating'].value_counts().sort_index())

Found cached dataset amazon_reviews_multi (C:/Users/User/.cache/huggingface/datasets/mteb___amazon_reviews_multi/en/1.0.0/61154b14772a2d73f3554caa8f5bd1fec4b65ec64f70bb93fa9fec6b038a4355)
Loading cached shuffled indices for dataset at C:\Users\User\.cache\huggingface\datasets\mteb___amazon_reviews_multi\en\1.0.0\61154b14772a2d73f3554caa8f5bd1fec4b65ec64f70bb93fa9fec6b038a4355\cache-7f3c2b000fc47742.arrow
Found cached dataset amazon_reviews_multi (C:/Users/User/.cache/huggingface/datasets/mteb___amazon_reviews_multi/fr/1.0.0/61154b14772a2d73f3554caa8f5bd1fec4b65ec64f70bb93fa9fec6b038a4355)
Loading cached shuffled indices for dataset at C:\Users\User\.cache\huggingface\datasets\mteb___amazon_reviews_multi\fr\1.0.0\61154b14772a2d73f3554caa8f5bd1fec4b65ec64f70bb93fa9fec6b038a4355\cache-dec31a132c0e1c2c.arrow
Found cached dataset amazon_reviews_multi (C:/Users/User/.cache/huggingface/datasets/mteb___amazon_reviews_multi/de/1.0.0/61154b14772a2d73f3554caa8f5bd1fec4b65ec64f70bb93fa9fec6b038a43

Downloaded 80 real reviews across 4 languages

Star rating distribution:
star_rating
1    20
2    12
3    16
4    24
5     8
Name: count, dtype: int64


## Initial Sampling Issue

Two initial download attempts using streaming mode returned entirely 
negative reviews. The first attempt (60 reviews, no shuffle) and a 
second attempt adding `.shuffle()` before `.take()` (80 reviews) both 
returned only 1-star ratings across all four languages, despite the 
dataset documentation confirming a balanced 5-label distribution. The 
issue and resolution are detailed below


## Sampling Method Correction

Initial attempts using streaming mode with `.take()` after shuffling 
returned exclusively 1-star reviews across all languages, despite the 
underlying dataset being balanced across 5 rating levels. This was 
caused by the streaming shuffle buffer not adequately randomising 
within the underlying file structure, which appears grouped by label.

Switching to direct dataset loading with `.shuffle()` and `.select()` 
resolved the issue, producing a properly distributed sample across 
all five star ratings (1-star: 20, 2-star: 12, 3-star: 16, 4-star: 24, 
5-star: 8).

## Convert Ratings to Sentiment Labels
Mapping star ratings to sentiment categories using the same logic 
established in our original dataset, ratings 1-2 as negative, 
3 as neutral, 4-5 as positive.

In [10]:
def rating_to_sentiment(rating):
    if rating <= 2:
        return 'negative'
    elif rating == 3:
        return 'neutral'
    else:
        return 'positive'

real_world_df['true_sentiment'] = real_world_df['star_rating'].apply(rating_to_sentiment)

print("Sentiment distribution in real-world sample:")
print(real_world_df['true_sentiment'].value_counts())

Sentiment distribution in real-world sample:
true_sentiment
negative    32
positive    32
neutral     16
Name: count, dtype: int64


## Validate Classical Models Against Real-World Data
Loading our saved Naive Bayes and Logistic Regression models and 
the fitted TF-IDF vectoriser to test genuine generalisation 
performance on real customer reviews.

In [13]:
import pickle
import re

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Load saved models and vectoriser
with open(os.path.join(PROJECT_ROOT, 'models', 'naive_bayes_model.pkl'), 'rb') as f:
    nb_model = pickle.load(f)

with open(os.path.join(PROJECT_ROOT, 'models', 'logistic_regression_model.pkl'), 'rb') as f:
    lr_model = pickle.load(f)

with open(os.path.join(PROJECT_ROOT, 'data', 'processed', 'tfidf_vectoriser.pkl'), 'rb') as f:
    tfidf = pickle.load(f)



## Generate Predictions
Cleaning real-world review text using the same preprocessing logic 
applied to our training data, then generating predictions from 
both classical models.

In [14]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[!]{2,}', '!', text)
    text = re.sub(r'[?]{2,}', '?', text)
    text = re.sub(r'[.]{2,}', '.', text)
    text = re.sub(r'[^a-zA-ZÀ-ÿ0-9\s\.,!?\'-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

real_world_df['cleaned_text'] = real_world_df['review_text'].apply(clean_text)

# Transform using our fitted TF-IDF vectoriser
X_real = tfidf.transform(real_world_df['cleaned_text'])

# Predict
real_world_df['nb_prediction'] = nb_model.predict(X_real)
real_world_df['lr_prediction'] = lr_model.predict(X_real)

# Evaluate
from sklearn.metrics import f1_score, accuracy_score

nb_f1 = f1_score(real_world_df['true_sentiment'], real_world_df['nb_prediction'], average='weighted')
lr_f1 = f1_score(real_world_df['true_sentiment'], real_world_df['lr_prediction'], average='weighted')

print("REAL-WORLD VALIDATION RESULTS")
print(f"Naive Bayes weighted F1:        {nb_f1:.4f}")
print(f"Logistic Regression weighted F1: {lr_f1:.4f}")

print(f"\nSample predictions vs true labels:")
print(real_world_df[['language', 'true_sentiment', 'nb_prediction', 'lr_prediction']].head(15))

REAL-WORLD VALIDATION RESULTS
Naive Bayes weighted F1:        0.4806
Logistic Regression weighted F1: 0.4682

Sample predictions vs true labels:
   language true_sentiment nb_prediction lr_prediction
0        en       negative      positive      positive
1        en       negative      negative      negative
2        en       positive      positive      positive
3        en       positive       neutral      positive
4        en       negative      negative      negative
5        en       negative      negative      negative
6        en        neutral       neutral       neutral
7        en       positive      positive      positive
8        en        neutral      positive      positive
9        en        neutral      positive      positive
10       en       positive      positive      positive
11       en       positive       neutral       neutral
12       en       negative      positive      positive
13       en       positive      positive      positive
14       en       negative    

## Real-World Validation (Critical Finding)
Performance collapses dramatically when classical models trained on 
the synthetic ShopEase dataset are tested against genuine customer 
reviews. Weighted F1 drops from a perfect 1.0000 on the synthetic 
test set to 0.4806 (Naive Bayes) and 0.4682 (Logistic Regression) 
on real Amazon reviews, performance only marginally better than 
random guessing for a three-class problem.

This confirms definitively that the perfect scores reported in 
notebook 07 reflect memorisation of a finite template vocabulary 
rather than genuine sentiment understanding. When exposed to authentic 
linguistic variation, sarcasm, mixed sentiment within a single review, 
and vocabulary outside the 444-template pool, both models fail to 
generalise meaningfully.

**This is the most important methodological insight in this project.** 
It demonstrates that high reported accuracy is meaningless without 
validation against representative, real-world data. Any production 
deployment of these classical models would require retraining on 
genuine customer feedback before deployment could be responsibly 
recommended.

This finding directly motivates the use of XLM-RoBERTa in notebook 09, 
a pretrained transformer model with broad language understanding 
developed from massive real-world multilingual corpora, which we 
expect to generalise substantially better than TF-IDF-based classical 
models despite never having seen ShopEase-specific data.